Simple text-to-speech with Google Text to Speech

In [ ]:
!pip install gTTS

In [ ]:
from gtts import gTTS

text_to_say = "This is a sample piece of text read by gTTS."

language = "en"
gtts_object = gTTS(text = text_to_say,
     lang = language,
     slow = False) #the language is set to English and we do not want the text read slowly.

gtts_object.save("/content/gtts.wav") #saving the text to speech audio file to our google drive

In [ ]:
from IPython.display import Audio

Audio("/content/gtts.wav")

Text to speech with PyTorch, Tacotron 2 and WaveGlow


In [ ]:
import torch

cuda_is_available = torch.cuda.is_available()
device = torch.device("cuda" if cuda_is_available else "cpu")

def load_waveglow():
    waveglow = torch.hub.load("nvidia/DeepLearningExamples:torchhub",
                              "nvidia_waveglow")

    waveglow = waveglow.remove_weightnorm(waveglow)
    waveglow = waveglow.to(device)
    waveglow.eval()
    return waveglow




In [ ]:
waveglow = load_waveglow()

/usr/local/lib/python3.10/dist-packages/torch/hub.py:294: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/nvidia/DeepLearningExamples/zipball/torchhub" to /root/.cache/torch/hub/torchhub.zip
/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/common.py:13: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn

In [ ]:
#download and run tacotron2

tacotron2 = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub",
                           "nvidia_tacotron2",
                           model_match = "fp16")
tacotron2 = tacotron2.to(device)
tacotron2.eval()

Downloading: "https://github.com/NVIDIA/DeepLearningExamples/zipball/torchhub" to /root/.cache/torch/hub/torchhub.zip


Tacotron2(
  (embedding): Embedding(148, 512)
  (encoder): Encoder(
    (convolutions): ModuleList(
      (0-2): 3 x Sequential(
        (0): ConvNorm(
          (conv): Conv1d(512, 512, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (lstm): LSTM(512, 256, batch_first=True, bidirectional=True)
  )
  (decoder): Decoder(
    (prenet): Prenet(
      (layers): ModuleList(
        (0): LinearNorm(
          (linear_layer): Linear(in_features=80, out_features=256, bias=False)
        )
        (1): LinearNorm(
          (linear_layer): Linear(in_features=256, out_features=256, bias=False)
        )
      )
    )
    (attention_rnn): LSTMCell(768, 1024)
    (attention_layer): Attention(
      (query_layer): LinearNorm(
        (linear_layer): Linear(in_features=1024, out_features=128, bias=False)
      )
      (memory_layer): LinearNorm(
        (linear_layer): Linear(in_fea

In [ ]:
#create sample text to be used for text-to-speech

text = "This is sample piece of text converted into speech by Waveglow and Tacotron2"
!pip install unidecode

utils = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub",
                       "nvidia_tts_utils")
sequences, lengths = utils.prepare_input_sequence([text])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 5.6 MB/s eta 0:00:00


Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub
/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/SpeechSynthesis/Tacotron2/tacotron2/text/__init__.py:74: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  return s in _symbol_to_id and s is not '_' and s is not '~'
/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/SpeechSynthesis/Tacotron2/tacotron2/text/__init__.py:74: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  return s in _symbol_to_id and s is not '_' and s is not '~'


In [ ]:
sequences

tensor([[57, 45, 46, 56, 11, 46, 56, 11, 56, 38, 50, 53, 49, 42, 11, 53, 46, 42,
         40, 42, 11, 52, 43, 11, 57, 42, 61, 57, 11, 40, 52, 51, 59, 42, 55, 57,
         42, 41, 11, 46, 51, 57, 52, 11, 56, 53, 42, 42, 40, 45, 11, 39, 62, 11,
         60, 38, 59, 42, 44, 49, 52, 60, 11, 38, 51, 41, 11, 57, 38, 40, 52, 57,
         55, 52, 51, 57, 60, 52]], device='cuda:0')

In [ ]:
lengths

tensor([78], device='cuda:0')

In [ ]:
with torch.no_grad():

  mel, _, _ = tacotron2.infer(sequences, lengths)
  audio = waveglow.infer(mel)
  print(audio)

tensor([[ 6.8944e-05,  4.5786e-04,  1.6473e-03,  ..., -9.7706e-05,
         -2.0456e-04, -3.5975e-04]], device='cuda:0')


In [ ]:
print(audio[0])

tensor([ 6.8944e-05,  4.5786e-04,  1.6473e-03,  ..., -9.7706e-05,
        -2.0456e-04, -3.5975e-04], device='cuda:0')


In [ ]:
print(audio[0].data)

tensor([ 6.8944e-05,  4.5786e-04,  1.6473e-03,  ..., -9.7706e-05,
        -2.0456e-04, -3.5975e-04], device='cuda:0')


In [ ]:
audio_numpy = audio[0].data.cpu().numpy()
print(audio_numpy)

[ 6.8944239e-05  4.5786484e-04  1.6473127e-03 ... -9.7705677e-05
 -2.0456365e-04 -3.5974922e-04]


In [ ]:
rate = 22050

from scipy.io.wavfile import write
write("textToSpeechWithTorch.wav", rate, audio_numpy)


In [ ]:
from IPython.display import Audio

Audio(audio_numpy, rate = rate)


Text to speech with pyttsx3

In [ ]:
!pip install pyttsx3

In [ ]:
import pyttsx3 as tts

!sudo apt update
!sudo apt install espeak
!sudo apt install ffmpeg

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:10 http://security.ubuntu.com/ubuntu jammy-security InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
40 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
e

In [ ]:
engine = tts.init()
engine

In [ ]:
voices = engine.getProperty("voices")

for voice in voices:
  print(voices, "\n", voice.id)

[<pyttsx3.voice.Voice object at 0x79fbc3685f90>, <pyttsx3.voice.Voice object at 0x79fbc3685f60>, <pyttsx3.voice.Voice object at 0x79fbc36878b0>, <pyttsx3.voice.Voice object at 0x79fbc3687a90>, <pyttsx3.voice.Voice object at 0x79fbc3686bf0>, <pyttsx3.voice.Voice object at 0x79fbc3685ff0>, <pyttsx3.voice.Voice object at 0x79fbc3686b90>, <pyttsx3.voice.Voice object at 0x79fbc3687850>, <pyttsx3.voice.Voice object at 0x79fbc3686fe0>, <pyttsx3.voice.Voice object at 0x79fbc3686ce0>, <pyttsx3.voice.Voice object at 0x79fbc3686b30>, <pyttsx3.voice.Voice object at 0x79fbc3686a70>, <pyttsx3.voice.Voice object at 0x79fbc36865c0>, <pyttsx3.voice.Voice object at 0x79fbc36864d0>, <pyttsx3.voice.Voice object at 0x79fbc3686470>, <pyttsx3.voice.Voice object at 0x79fbc3686620>, <pyttsx3.voice.Voice object at 0x79fbc36870d0>, <pyttsx3.voice.Voice object at 0x79fbc3687190>, <pyttsx3.voice.Voice object at 0x79fbc3687220>, <pyttsx3.voice.Voice object at 0x79fbc3687550>, <pyttsx3.voice.Voice object at 0x79fbc3

In [ ]:
#filter for english voices
for voice in voices:
    if voice.name.startswith("en"):
      print(voice)


<Voice id=english
          name=english
          languages=[b'\x02en-gb']
          gender=male
          age=None>
<Voice id=en-scottish
          name=en-scottish
          languages=[b'\x05en-sc']
          gender=male
          age=None>
<Voice id=english-north
          name=english-north
          languages=[b'\x05en-uk-north']
          gender=male
          age=None>
<Voice id=english_rp
          name=english_rp
          languages=[b'\x05en-uk-rp']
          gender=male
          age=None>
<Voice id=english_wmids
          name=english_wmids
          languages=[b'\x05en-uk-wmids']
          gender=male
          age=None>
<Voice id=english-us
          name=english-us
          languages=[b'\x02en-us']
          gender=male
          age=None>
<Voice id=en-westindies
          name=en-westindies
          languages=[b'\x05en-wi']
          gender=male
          age=None>


In [ ]:
engine.getProperty("rate")

175

In [ ]:
engine.setProperty("rate", 100)

In [ ]:
text = "This is a sample piece of text read by TTS"
engine.save_to_file(text, "textToSpeechWithTTS.mp3")
engine.runAndWait()

In [ ]:
from IPython.display import Audio

Audio("/content/textToSpeechWithTTS.mp3")